# Authenticated Product Data Extraction for Procurement Agents

Extract structured product data from any commerce URL with per-field confidence scores using ShopGraph's authenticated extraction API. Confidence-aware routing separates verified data from fields that need human review.

In [ ]:
%pip install requests langchain langchain-openai

In [ ]:
import os
import json
import requests

SHOPGRAPH_API_KEY = os.environ.get("SHOPGRAPH_API_KEY", "your-api-key")
SHOPGRAPH_URL = "https://shopgraph.dev/api/enrich"

In [ ]:
def extract_product(url: str) -> dict:
    """Extract structured product data with confidence scores."""
    response = requests.post(
        SHOPGRAPH_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {SHOPGRAPH_API_KEY}",
        },
        json={"url": url},
    )

    # Error handling: return structured error, don't raise
    if response.status_code != 200:
        return {"error": True, "status_code": response.status_code, "message": f"Extraction failed for {url}"}

    data = response.json()
    if "product" not in data:
        return {"error": True, "status_code": response.status_code, "message": f"No product data returned for {url}"}

    return data


data = extract_product("https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8")

# Response structure:
# data["product"]["product_name"]  -> "DAYTON 1/2 HP Jet Pump, Model 5UXK1"
# data["product"]["price"]["amount"]  -> 284.00
# data["product"]["price"]["currency"]  -> "USD"
# data["product"]["availability"]  -> "in_stock"
# data["product"]["confidence"]["overall"]  -> 0.93
# data["product"]["confidence"]["per_field"]["price"]  -> 0.93
# data["product"]["_shopgraph"]["field_confidence"]["price"]  -> 0.93
# data["product"]["_shopgraph"]["field_freshness"]["price"]["decayed"]  -> False
# data["cached"]  -> False
# data["credit_mode"]  -> "standard"

print(f"Product: {data.get('product', {}).get('product_name', 'N/A')}")

## Confidence-Aware Routing

Per-field confidence scores let you programmatically route data into different handling paths:

- **Verified** (confidence >= threshold): Safe for automation
- **Needs review** (confidence >= 0.50 but < threshold): Flag for human check
- **Missing** (confidence < 0.50 or absent): Request manual entry

This is the pattern that separates a brittle pipeline from a reliable one.

In [ ]:
def extract_with_confidence_routing(
    url: str, confidence_threshold: float = 0.8
) -> dict:
    """
    Extract product data and route fields into verified/review/missing buckets
    based on per-field confidence scores.
    """
    data = extract_product(url)

    # If extraction returned an error, pass it through
    if "error" in data:
        return data

    product = data["product"]
    meta = product["_shopgraph"]

    verified = {}
    needs_review = {}
    missing = {}

    field_map = {
        "product_name": product.get("product_name"),
        "brand": product.get("brand"),
        "description": product.get("description"),
        "price": product.get("price", {}).get("amount") if product.get("price") else None,
        "currency": product.get("price", {}).get("currency") if product.get("price") else None,
        "availability": product.get("availability"),
        "categories": product.get("categories"),
        "primary_image_url": product.get("primary_image_url"),
        "material": product.get("material"),
    }

    for field_name, value in field_map.items():
        if value is None or value == [] or value == "unknown":
            missing[field_name] = {"value": value, "reason": "not_available"}
            continue

        # Look up confidence from _shopgraph.field_confidence
        confidence = meta["field_confidence"].get(field_name, 0)

        entry = {"value": value, "confidence": confidence}

        # Check freshness for real-time fields
        freshness = meta.get("field_freshness", {}).get(field_name)
        if freshness and freshness.get("decayed"):
            entry["decayed"] = True
            needs_review[field_name] = entry
        elif confidence >= confidence_threshold:
            verified[field_name] = entry
        else:
            needs_review[field_name] = entry

    return {
        "url": product["url"],
        "extraction_method": meta["extraction_method"],
        "data_source": meta["data_source"],
        "overall_confidence": product["confidence"]["overall"],
        "verified": verified,
        "needs_review": needs_review,
        "missing": missing,
    }

In [ ]:
result = extract_with_confidence_routing(
    "https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8",
    confidence_threshold=0.85,
)

if "error" in result:
    print(f"Error: {result['message']}")
else:
    print("Verified fields:")
    for field, info in result["verified"].items():
        print(f"  {field}: {info['value']} (confidence: {info['confidence']})")

    print("\nNeeds review:")
    for field, info in result["needs_review"].items():
        print(f"  {field}: {info['value']} (confidence: {info['confidence']})")

    print("\nMissing:")
    for field, info in result["missing"].items():
        print(f"  {field}: {info['reason']}")

## Using with a LangChain Agent

Wrap the confidence-routed extraction as a LangChain tool so an agent can extract and reason about product data quality.

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.prompts import ChatPromptTemplate


@tool
def enrich_product(url: str) -> str:
    """Extract structured product data with confidence scores from a product URL."""
    result = extract_with_confidence_routing(url, confidence_threshold=0.8)
    return json.dumps(result, indent=2)


llm = ChatOpenAI(model="gpt-4o")
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a procurement assistant. Use the enrich_product tool to "
            "extract product data. Only trust verified fields for purchase decisions. "
            "Flag fields in needs_review for human approval.",
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_openai_functions_agent(llm, [enrich_product], prompt)
executor = AgentExecutor(agent=agent, tools=[enrich_product], verbose=True)

result = executor.invoke(
    {"input": "Get product details for https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8"}
)
print(result["output"])

## Notes

- Playground: 50 calls/month, no signup required. Starter tier: $99/month for 10K calls with API key. See https://shopgraph.dev/pricing
- Confidence scores range from 0.0 to 1.0, based on extraction method (Schema.org: ~0.93 baseline, LLM: ~0.70 baseline)
- The `_shopgraph.field_freshness` metadata shows whether cached data has decayed — useful for real-time pricing decisions
- Full API documentation: [shopgraph.dev](https://shopgraph.dev)
- Check site extraction status before testing: https://shopgraph.dev/leaderboard
- AgentReady scoring: append `?include_score=true` to score a URL across 6 agent-readiness dimensions
- UCP-compatible output: append `?format=ucp` for Universal Commerce Protocol schema
- Server-side confidence filtering: append `?strict_confidence_threshold=0.85` to scrub low-confidence fields to null
